# Data Inspection and Cleaning of Shootings & Victims Datasets

- Location of the dataset of **Shootings occurances**: https://data.cityofnewyork.us/Public-Safety/Shootings-2006-Present-/5ucz-vwe8/about_data
- Location of the dataset of the **Victims** of these shootings: https://data.cityofnewyork.us/Public-Safety/Shooting-Victims-2006-Present-/pztn-9bne/about_data

## Issues
- **Daylight Saving times** - Some recorded shootings occured when clocks were moving forward or back (e.g. **2006-10-29 01:35:00**)
- We cannot infer the time from this value because at 2am, the clocks go back to 1am. This means 01:35 could mean 1:35am the first time or 1:35am the second time - There is no way of knowing!
- Also, when clocks move forward, clock skips directly from 02:00 to 03:00 - Did the officer recording the time take this into consideration
- **Possible solution** aggegrate shooting occurances over a 3 hour period rather than 1 hour period 

In [6]:
%matplotlib inline
#imports required to see if sun has set/risen for a particular datetime in df_shootings
from astral import LocationInfo
import zoneinfo
import datetime
from astral.sun import sun

import pandas as pd
import geopandas as gpd #For plotting shooting locations
from shapely.geometry import Point
import matplotlib.pyplot as plt
shootings = "../data/Shootings_2006-Present_20260608.csv"
victims = "../data/Shooting_Victims_2006-Present_20260608.csv"

#To aid in sunrise and sunset times below
timezone = zoneinfo.ZoneInfo("America/New_York")
city = LocationInfo("New York", "United States", "America/New_York", 40.71, -74.01) #lat and long of NYC

df_shootings = pd.read_csv(shootings)
df_victims = pd.read_csv(victims)

## Shooting Occurrences

In [7]:
df_shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,04/20/2020,19:36:00,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301.0,192923.0,-73.942118,40.696199
1,139554478,11/07/2014,03:20:00,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185.0,241494.0,-73.899059,40.829485
2,167619814,08/01/2017,00:35:00,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360.0,261043.0,-73.833250,40.883065
3,212516311,04/29/2020,15:30:00,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085.0,248078.0,-73.910236,40.847565
4,270150948,06/20/2023,23:41:00,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925.0,209948.0,-73.925390,40.742919


- **Latitude and Longitude are the wrong way round in Raw dataset**

In [8]:
# swapping latitude and longitude columns
df_shootings[['Latitude', 'Longitude']] = df_shootings[['Longitude', 'Latitude']].values
df_shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
0,212200405,04/20/2020,19:36:00,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301.0,192923.0,40.696199,-73.942118
1,139554478,11/07/2014,03:20:00,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185.0,241494.0,40.829485,-73.899059
2,167619814,08/01/2017,00:35:00,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360.0,261043.0,40.883065,-73.833250
3,212516311,04/29/2020,15:30:00,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085.0,248078.0,40.847565,-73.910236
4,270150948,06/20/2023,23:41:00,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925.0,209948.0,40.742919,-73.925390


In [9]:
df_shootings.info()

<class 'pandas.DataFrame'>
RangeIndex: 24127 entries, 0 to 24126
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   INCIDENT_KEY        24127 non-null  int64  
 1   OCCUR_DATE          24127 non-null  str    
 2   OCCUR_TIME          24127 non-null  str    
 3   BORO                24127 non-null  str    
 4   LOC_OF_OCCUR_DESC   24127 non-null  str    
 5   PRECINCT            24127 non-null  int64  
 6   JURISDICTION_CODE   24125 non-null  float64
 7   LOC_CLASSFCTN_DESC  24094 non-null  str    
 8   LOCATION_DESC       9693 non-null   str    
 9   X_COORD_CD          24126 non-null  float64
 10  Y_COORD_CD          24126 non-null  float64
 11  Latitude            24126 non-null  float64
 12  Longitude           24126 non-null  float64
dtypes: float64(5), int64(2), str(6)
memory usage: 2.4 MB


- Missing values in multiple columns **except**
    - INCIDENT_KEY,
    - OCCUR_DATE,
    - OCCUR_TIME,
    - BORO,
    - LOC_OF_OCCUR_DESC
    - and PRECINCT

## Victims Dataset

In [10]:
df_victims.head()

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
0,299453927,299453927-1,25-44,MALE,BLACK,N
1,212336415,212336415-1,18-24,MALE,BLACK,N
2,280768895,280768895-1,25-44,MALE,BLACK,N
3,141824677,141824677-27687,25-44,MALE,BLACK HISPANIC,N
4,270079203,270079203-1,45-64,MALE,BLACK,Y


In [11]:
df_victims.info()

<class 'pandas.DataFrame'>
RangeIndex: 28912 entries, 0 to 28911
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   INCIDENT_KEY      28912 non-null  int64
 1   VICTIM_ID         28911 non-null  str  
 2   VICTIM_AGE_GROUP  28911 non-null  str  
 3   VICTIM_SEX        28911 non-null  str  
 4   VICTIM_RACE       28911 non-null  str  
 5   STAT_MURDER_FLG   28911 non-null  str  
dtypes: int64(1), str(5)
memory usage: 1.3 MB


- A missing value in all columns except INCIDENT_KEY

## Duplicates?

In [12]:
print(f"Number of duplicated rows in df_shootings: {df_shootings.duplicated().sum()}")
print(f"Number of duplicated rows in df_victims: {df_victims.duplicated().sum()}")

Number of duplicated rows in df_shootings: 0
Number of duplicated rows in df_victims: 0


In [13]:
#Shouldn't be any duplicate INCIDENT_KEY for shooting occurances
print(f"Number of duplicate INCIDENT_KEY in df_shootings: {df_shootings['INCIDENT_KEY'].duplicated().sum()}")

#There are probably multiple victims per shooting incident.
print(f"Number of duplicate INCIDENT_KEY in df_victims: {df_victims['INCIDENT_KEY'].duplicated().sum()}")

#But, VICTIM_ID should be unique (zero)
print(f"Number of duplicate VICTIM_ID's in df_victims: {df_victims["VICTIM_ID"].duplicated().sum()}")

Number of duplicate INCIDENT_KEY in df_shootings: 0
Number of duplicate INCIDENT_KEY in df_victims: 4783
Number of duplicate VICTIM_ID's in df_victims: 0


## Distinct Categorical Variables (Shootings & Victims)

In [14]:
df_shootings["BORO"].unique()

<StringArray>
['BROOKLYN', 'BRONX', 'QUEENS', 'MANHATTAN', 'STATEN ISLAND']
Length: 5, dtype: str

In [15]:
df_shootings["LOC_OF_OCCUR_DESC"].unique()

<StringArray>
['OUTSIDE', 'INSIDE']
Length: 2, dtype: str

In [16]:
df_shootings["PRECINCT"].unique()

array([ 79,  42,  47,  46, 108,  70, 114,  32,  52, 120,  34,  23, 104,
        71,  83,  90,  25, 115, 103,  75, 113,  44, 109,  10,  33,  77,
        61,  73,  28,  67,  43,  76,  81,  41, 105,  63,  60,  88, 122,
        48, 106,  69, 102,  50,  45,  40, 107,  49,  30,  62, 100,   9,
       101,  72,  24,  78,  68,  18, 110,  66,  19,  20,  84,   6,   7,
        26,   5,  94, 123, 111,  13,  14, 112,  17,   1, 121,  22, 116])

In [17]:
print(f"There are {len(df_shootings["PRECINCT"].unique())} unique Precincts in this df_shootings")

There are 78 unique Precincts in this df_shootings


NYC is divided into 78 precints: https://www.nyc.gov/site/nypd/bureaus/patrol/find-your-precinct.page

In [18]:
df_shootings["JURISDICTION_CODE"].unique()

array([ 2.,  0.,  1., nan])

- Jurisdiction where the shooting incident occurred. 
- Jurisdiction codes 0(Patrol), 1(Transit) and 2(Housing) represent NYPD whilst codes 3 and more represent non NYPD jurisdictions
- https://data.cityofnewyork.us/Public-Safety/Shootings-2006-Present-/5ucz-vwe8/about_data

In [19]:
df_shootings["LOC_CLASSFCTN_DESC"].unique()

<StringArray>
[    'HOUSING',    'DWELLING',      'STREET',  'COMMERCIAL', 'TAXI/LIVERY',
     'VEHICLE',           nan,  'PLAYGROUND',       'OTHER', 'PARKING LOT',
     'TRANSIT']
Length: 11, dtype: str

In [20]:
df_shootings["LOCATION_DESC"].unique()

<StringArray>
['MULTI DWELL - PUBLIC HOUS',                 'PVT HOUSE',
                         nan,   'MULTI DWELL - APT BUILD',
            'BAR/NIGHT CLUB',         'FACTORY/WAREHOUSE',
                      'NONE',       'DRY CLEANER/LAUNDRY',
           'COMMERCIAL BLDG',            'GROCERY/BODEGA',
          'RESTAURANT/DINER',         'BEAUTY/NAIL SALON',
 'SOCIAL CLUB/POLICY LOCATI',               'HOTEL/MOTEL',
                 'FAST FOOD',               'GAS STATION',
             'JEWELRY STORE',               'SUPERMARKET',
               'CHAIN STORE',         'CLOTHING BOUTIQUE',
            'SMALL MERCHANT',        'STORE UNCLASSIFIED',
             'VARIETY STORE',              'LIQUOR STORE',
                    'SCHOOL',                'DEPT STORE',
               'CANDY STORE',          'STORAGE FACILITY',
                'DRUG STORE',               'VIDEO STORE',
                'SHOE STORE',           'TELECOMM. STORE',
      'GYM/FITNESS FACILITY',             

In [21]:
df_victims["VICTIM_AGE_GROUP"].unique()

<StringArray>
['25-44', '18-24', '45-64', '<18', 'UNKNOWN', '65+', '1022', nan]
Length: 8, dtype: str

In [22]:
df_victims["VICTIM_SEX"].unique()

<StringArray>
['MALE', 'FEMALE', 'UNKNOWN', 'INTERSEX', nan]
Length: 5, dtype: str

In [23]:
df_victims["VICTIM_RACE"].unique()

<StringArray>
[                         'BLACK',                 'BLACK HISPANIC',
                 'WHITE HISPANIC',                          'WHITE',
       'ASIAN / PACIFIC ISLANDER',                        'UNKNOWN',
 'AMERICAN INDIAN/ALASKAN NATIVE',                              nan]
Length: 8, dtype: str

In [24]:
df_victims["STAT_MURDER_FLG"].unique()

<StringArray>
['N', 'Y', nan]
Length: 3, dtype: str

## Closer look at Missing Values

In [25]:
#Let's look at % of missing vals per column
df_shootings.isna().mean()*100

INCIDENT_KEY           0.000000
OCCUR_DATE             0.000000
OCCUR_TIME             0.000000
BORO                   0.000000
LOC_OF_OCCUR_DESC      0.000000
PRECINCT               0.000000
JURISDICTION_CODE      0.008289
LOC_CLASSFCTN_DESC     0.136776
LOCATION_DESC         59.825092
X_COORD_CD             0.004145
Y_COORD_CD             0.004145
Latitude               0.004145
Longitude              0.004145
dtype: float64

In [26]:
#Only one missing row for Lat and Long
df_shootings[
(df_shootings["Latitude"].isna()) |
(df_shootings["Longitude"].isna())
 ]

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
22113,322313941,03/24/2026,02:14:00,BROOKLYN,OUTSIDE,61,0.0,STREET,NaN,NaN,NaN,NaN,NaN


In [27]:
df_shootings[ df_shootings["JURISDICTION_CODE"].isna() ]

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
1967,33110745,07/13/2007,01:10:00,QUEENS,INSIDE,104,NaN,COMMERCIAL,SOCIAL CLUB/POLICY LOCATI,1009103.0,194183.0,40.699638,-73.910371
16697,194525846,03/09/2019,02:41:00,MANHATTAN,OUTSIDE,25,NaN,STREET,NaN,1000455.0,230842.0,40.800277,-73.941471


In [28]:
df_shootings[df_shootings["LOC_CLASSFCTN_DESC"].isna()]

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude
48,16814040,06/19/2006,00:52:00,MANHATTAN,OUTSIDE,28,0.0,NaN,NaN,996959.0,232155.0,40.803887,-73.954096
1245,33110760,07/15/2007,19:40:00,BROOKLYN,OUTSIDE,67,0.0,NaN,PVT HOUSE,999032.0,176603.0,40.651407,-73.946730
1399,28620014,04/21/2007,00:35:00,MANHATTAN,OUTSIDE,34,0.0,NaN,MULTI DWELL - APT BUILD,1003586.0,251293.0,40.856403,-73.930103
1954,32320694,06/17/2007,23:32:00,MANHATTAN,OUTSIDE,28,0.0,NaN,NaN,996467.0,233050.0,40.806344,-73.955871
3985,33535836,08/03/2007,19:00:00,BROOKLYN,INSIDE,73,0.0,NaN,NaN,1010520.0,184517.0,40.673103,-73.905298
4054,46715566,05/24/2008,03:35:00,BRONX,OUTSIDE,47,0.0,NaN,BAR/NIGHT CLUB,1024046.0,261196.0,40.883515,-73.856083
4510,33725001,08/13/2007,02:10:00,BROOKLYN,OUTSIDE,88,0.0,NaN,MULTI DWELL - APT BUILD,992375.0,188772.0,40.684817,-73.970706
6996,73298761,06/20/2010,05:23:00,QUEENS,OUTSIDE,103,0.0,NaN,BAR/NIGHT CLUB,1046499.0,197760.0,40.709272,-73.775472
8555,81116166,10/10/2011,00:17:00,BROOKLYN,OUTSIDE,75,0.0,NaN,NaN,1015792.0,176133.0,40.650073,-73.886332
8651,86761907,09/16/2012,04:10:00,BROOKLYN,OUTSIDE,75,0.0,NaN,MULTI DWELL - APT BUILD,1013100.0,182830.0,40.668464,-73.896004


- LOCATION_DESC has 9693 non-null values or 14434 data values missing

In [29]:
#Let's look at % of missing vals per column
df_victims.isna().mean()*100

INCIDENT_KEY        0.000000
VICTIM_ID           0.003459
VICTIM_AGE_GROUP    0.003459
VICTIM_SEX          0.003459
VICTIM_RACE         0.003459
STAT_MURDER_FLG     0.003459
dtype: float64

In [30]:
#Hopefully only one row missing in victims dataframe
df_victims[
(df_victims["VICTIM_ID"].isna()) |
(df_victims["VICTIM_AGE_GROUP"].isna())|
(df_victims["VICTIM_SEX"].isna()) |
(df_victims["VICTIM_RACE"].isna())|
(df_victims["STAT_MURDER_FLG"].isna())   
 ]

,INCIDENT_KEY,VICTIM_ID,VICTIM_AGE_GROUP,VICTIM_SEX,VICTIM_RACE,STAT_MURDER_FLG
28911,203259656,NaN,NaN,NaN,NaN,NaN


In [31]:
#Cross reference with shootings dataframe
df_shootings[df_shootings["INCIDENT_KEY"] == 203259656]

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude


- This Incident_key (203259656) is not found in the shootings dataset.
- This Foreign Key Joins to Shootings using the INCIDENT_KEY field

## Map of all Shooting Incidents


In [32]:
#Isolate the lat and long into separate dataframe
df_coord = df_shootings[["Latitude", "Longitude"]].copy()
df_coord.head()

,Latitude,Longitude
0,40.696199,-73.942118
1,40.829485,-73.899059
2,40.883065,-73.833250
3,40.847565,-73.910236
4,40.742919,-73.925390


In [33]:
#still have one missing coordinate
df_coord.isna().sum()

Latitude     1
Longitude    1
dtype: int64

In [34]:
#Let's remove this and create a cleaned dataset of just coordinates
df_coord_cleaned = df_coord.dropna()
df_coord_cleaned.isna().sum()

Latitude     0
Longitude    0
dtype: int64

In [35]:
#Need to create a GeoDataFrame for plotting on maps (Using GEoPandas)
gdf = gpd.GeoDataFrame(
    df_coord_cleaned, 
    geometry=gpd.points_from_xy(df_coord_cleaned.Longitude, df_coord_cleaned.Latitude), 
    crs="EPSG:4326"
)
gdf.head()

,Latitude,Longitude,geometry
0,40.696199,-73.942118,POINT (-73.94212 40.6962)
1,40.829485,-73.899059,POINT (-73.89906 40.82948)
2,40.883065,-73.833250,POINT (-73.83325 40.88306)
3,40.847565,-73.910236,POINT (-73.91024 40.84757)
4,40.742919,-73.925390,POINT (-73.92539 40.74292)


In [36]:
#Displaying locations of shootings
#Be aware that some data points on this graph could include multiple shooting incidents - will need to colour code this!
#Will explore in more depth while doing EDA
gdf.explore(
    tiles='OpenStreetMap'   
    #tiles='CartoDB positron'
)



## Fix Date Column in Shootings Dataset

In [37]:
#First need to check all date rows look like this: mm/dd/yyyy
#Then we'll convert the column into datetime

mask = df_shootings["OCCUR_DATE"].str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
invalid_dates = df_shootings.loc[~mask, "OCCUR_DATE"]
display(invalid_dates)

Series([], Name: OCCUR_DATE, dtype: str)

In [38]:
#Check time column 
mask= df_shootings["OCCUR_TIME"].str.match(r"^\d{2}:\d{2}:\d{2}$", na=False)
invalid_times = df_shootings.loc[~mask, "OCCUR_TIME"]
display(invalid_times)

Series([], Name: OCCUR_TIME, dtype: str)

- Both date and time formats are consistent for every row: mm/dd/yyyy and HH:MM:SS

In [39]:
#Combine date and time into new column and convert to datetime object
df_shootings["datetime"] = pd.to_datetime(df_shootings["OCCUR_DATE"] + " " + df_shootings["OCCUR_TIME"])
df_shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,datetime
0,212200405,04/20/2020,19:36:00,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301.0,192923.0,40.696199,-73.942118,2020-04-20 19:36:00
1,139554478,11/07/2014,03:20:00,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185.0,241494.0,40.829485,-73.899059,2014-11-07 03:20:00
2,167619814,08/01/2017,00:35:00,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360.0,261043.0,40.883065,-73.833250,2017-08-01 00:35:00
3,212516311,04/29/2020,15:30:00,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085.0,248078.0,40.847565,-73.910236,2020-04-29 15:30:00
4,270150948,06/20/2023,23:41:00,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925.0,209948.0,40.742919,-73.925390,2023-06-20 23:41:00


In [40]:
df_shootings.info()

<class 'pandas.DataFrame'>
RangeIndex: 24127 entries, 0 to 24126
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   INCIDENT_KEY        24127 non-null  int64         
 1   OCCUR_DATE          24127 non-null  str           
 2   OCCUR_TIME          24127 non-null  str           
 3   BORO                24127 non-null  str           
 4   LOC_OF_OCCUR_DESC   24127 non-null  str           
 5   PRECINCT            24127 non-null  int64         
 6   JURISDICTION_CODE   24125 non-null  float64       
 7   LOC_CLASSFCTN_DESC  24094 non-null  str           
 8   LOCATION_DESC       9693 non-null   str           
 9   X_COORD_CD          24126 non-null  float64       
 10  Y_COORD_CD          24126 non-null  float64       
 11  Latitude            24126 non-null  float64       
 12  Longitude           24126 non-null  float64       
 13  datetime            24127 non-null  datetime64[us]
dtypes

In [41]:
#Adding Day of Week column
#Maybe more shootings occur of Fri nights and weekends?
df_shootings["day_of_week"] = df_shootings["datetime"].dt.day_name()
df_shootings.head()

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,datetime,day_of_week
0,212200405,04/20/2020,19:36:00,BROOKLYN,OUTSIDE,79,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1000301.0,192923.0,40.696199,-73.942118,2020-04-20 19:36:00,Monday
1,139554478,11/07/2014,03:20:00,BRONX,INSIDE,42,0.0,DWELLING,PVT HOUSE,1012185.0,241494.0,40.829485,-73.899059,2014-11-07 03:20:00,Friday
2,167619814,08/01/2017,00:35:00,BRONX,OUTSIDE,47,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1030360.0,261043.0,40.883065,-73.833250,2017-08-01 00:35:00,Tuesday
3,212516311,04/29/2020,15:30:00,BRONX,OUTSIDE,46,0.0,STREET,NaN,1009085.0,248078.0,40.847565,-73.910236,2020-04-29 15:30:00,Wednesday
4,270150948,06/20/2023,23:41:00,QUEENS,OUTSIDE,108,0.0,STREET,NaN,1004925.0,209948.0,40.742919,-73.925390,2023-06-20 23:41:00,Tuesday


In [42]:
# Add a column for whether it's dark or light out
#When Civil Twilight (Dawn) starts in morning, it should be bright enough to see without artificial light. As soon as we get morning civil twilight, we can call this "daytime"
#Night starts after Dusk or when civil twilight ends. After dusk, we'll label this "Nightime"
# Need to localise our datetime as well

# defining a function to check whether it's day or night
#Day means it's light enough to see without artificial light
#Night means artificial light is required to see.
def check_sun_period(row):
    dt = row['datetime']
    # Calculate sun times dynamically based on each row's specific date
    s = sun(city.observer, date=dt.date(), tzinfo=timezone)

    #Removing timezone tag to keep naive time because we'll get errors due to daylight saving time inconsistencies 
    dawn = s['dawn'].replace(tzinfo=None) 
    dusk = s['dusk'].replace(tzinfo=None)

    return 'Day' if dawn <= dt <= dusk else 'Night'

#Is it light enough to see or is it dark?
df_shootings['day_or_night'] = df_shootings.apply(check_sun_period, axis=1)


In [43]:
df_shootings.tail(10)

,INCIDENT_KEY,OCCUR_DATE,OCCUR_TIME,BORO,LOC_OF_OCCUR_DESC,PRECINCT,JURISDICTION_CODE,LOC_CLASSFCTN_DESC,LOCATION_DESC,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,datetime,day_of_week,day_or_night
24117,320602481,02/18/2026,09:30:00,BROOKLYN,INSIDE,60,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,989304.0,154751.0,40.591439,-73.981804,2026-02-18 09:30:00,Wednesday,Day
24118,320797107,02/22/2026,06:10:00,BROOKLYN,OUTSIDE,72,0.0,STREET,NaN,980200.0,175909.0,40.649514,-74.014597,2026-02-22 06:10:00,Sunday,Night
24119,320906869,02/25/2026,00:40:00,BRONX,OUTSIDE,49,0.0,STREET,NaN,1020955.0,247150.0,40.844977,-73.867338,2026-02-25 00:40:00,Wednesday,Night
24120,321102064,03/01/2026,04:30:00,QUEENS,OUTSIDE,103,0.0,STREET,NaN,1037846.0,193895.0,40.698720,-73.806714,2026-03-01 04:30:00,Sunday,Night
24121,321394673,03/06/2026,06:35:00,BRONX,INSIDE,42,2.0,HOUSING,MULTI DWELL - PUBLIC HOUS,1010922.0,239853.0,40.824985,-73.903629,2026-03-06 06:35:00,Friday,Day
24122,321420696,03/06/2026,22:28:00,MANHATTAN,OUTSIDE,25,0.0,STREET,NaN,1000558.0,231080.0,40.800930,-73.941098,2026-03-06 22:28:00,Friday,Night
24123,321624285,03/10/2026,19:31:00,BROOKLYN,OUTSIDE,79,0.0,HOUSING,MULTI DWELL - PUBLIC HOUS,995918.0,190890.0,40.690626,-73.957927,2026-03-10 19:31:00,Tuesday,Night
24124,322156241,03/20/2026,17:06:00,BROOKLYN,INSIDE,81,0.0,DWELLING,MULTI DWELL - APT BUILD,1001586.0,189839.0,40.687732,-73.937491,2026-03-20 17:06:00,Friday,Day
24125,322702777,03/31/2026,19:16:00,BRONX,OUTSIDE,48,0.0,STREET,NaN,1014085.0,250720.0,40.854801,-73.892152,2026-03-31 19:16:00,Tuesday,Day
24126,322710772,03/31/2026,22:22:00,MANHATTAN,OUTSIDE,28,0.0,STREET,NaN,997029.0,230313.0,40.798831,-73.953846,2026-03-31 22:22:00,Tuesday,Night


- Do shootings Occur more when LOC_OF_OCCUR_DESC = "OUTSIDE" when day_or_night="Night"?
- Are there more shootings when it's naturally more dark?

LOCATION_DESC has about 60% of it's data missing from shootings_df. However, LOC_CLASSFCTN_DESC has only 0.14% missing. Because both these features are linked, maybe we can fill in the missing values in LOC_CLASSFCTN_DESC 

In [45]:
#If LOCATION_DESC = BAR/NIGHT CLUB, how is this represented in the LOC_CLASSFCTN_DESC?
#Is this classification consistent or not?
df_shootings.groupby("LOCATION_DESC")["LOC_CLASSFCTN_DESC"].unique()

LOCATION_DESC
ATM                                                                  [HOUSING]
BANK                                                      [COMMERCIAL, STREET]
BAR/NIGHT CLUB               [STREET, COMMERCIAL, OTHER, DWELLING, nan, PAR...
BEAUTY/NAIL SALON                               [COMMERCIAL, STREET, DWELLING]
CANDY STORE                                               [COMMERCIAL, STREET]
CHAIN STORE                      [COMMERCIAL, PARKING LOT, PLAYGROUND, STREET]
CHECK CASH                                                        [COMMERCIAL]
CLOTHING BOUTIQUE                                         [STREET, COMMERCIAL]
COMMERCIAL BLDG              [STREET, COMMERCIAL, PARKING LOT, DWELLING, OT...
DEPT STORE                                           [COMMERCIAL, PARKING LOT]
DOCTOR/DENTIST                                            [COMMERCIAL, STREET]
DRUG STORE                                    [STREET, PLAYGROUND, COMMERCIAL]
DRY CLEANER/LAUNDRY                   

- We can see that there is no consistent way of categorising the locations. For example, a DRUG STORE has been also recorded as a [STREET, PLAYGROUND, COMMERCIAL]
- Maybe LOC_CLASSFCTN_DESC offers information on what's surrounding the crime scene? So, Crime happened in Drug store but this drug store is next to a playground.
- Maybe for this data analysis, it's more important to focuse on coordinates of shooting event and not the exact description of the crime scene location. There is simply too much missing data here and no consistent reporting.
- However, Only 0.14% of LOC_CLASSFCTN_DESC is missing and it could show us interesting patterns!
- I want to add weather data as well. Weather might effect whether someone chooses to go out or stay inside etc.

## Save Modified df_shootings to csv

In [46]:
df_shootings.to_csv("../data/Shootings_2006-Present_20260608_CLEANED.csv", index=False)